# Project Sandy — Launch Notebook

Run the cells below in order to start the **Django API** (port 3000) and **Vite frontend** (port 5173).

| Service | URL |
|---------|-----|
| Frontend | http://localhost:5173 |
| API | http://localhost:3000/api/ |

**Requirements:** Python 3, Node.js/npm, dependencies installed (`pip install -r requirements.txt`, `npm install`).

On Windows, each server opens in its own console window. Use the **Stop servers** cell when you are done.

In [ ]:
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import webbrowser
from pathlib import Path

# ── Project paths ──────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "manage.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "scripts" / "launch.ipynb").exists() or NOTEBOOK_DIR.name == "scripts":
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scripts" else NOTEBOOK_DIR
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if not (PROJECT_ROOT / "manage.py").exists():
    raise FileNotFoundError(
        f"Could not find manage.py. Open this notebook from the project root or scripts/ folder.\n"
        f"Current directory: {NOTEBOOK_DIR}"
    )

VENV_PYTHON = PROJECT_ROOT / "venv" / "Scripts" / "python.exe"
if not VENV_PYTHON.exists():
    VENV_PYTHON = PROJECT_ROOT / "venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON if VENV_PYTHON.exists() else sys.executable)

BACKEND_URL = "http://localhost:3000/api/"
FRONTEND_URL = "http://localhost:5173/"
BACKEND_PORT = 3000
FRONTEND_PORT = 5173

# Track subprocesses so we can stop them later
_processes: list[subprocess.Popen] = []


def _new_console_flags() -> int:
    if sys.platform == "win32" and hasattr(subprocess, "CREATE_NEW_CONSOLE"):
        return subprocess.CREATE_NEW_CONSOLE
    return 0


def start_process(name: str, command: list[str] | str, *, shell: bool = False) -> subprocess.Popen:
    print(f"Starting {name}...")
    proc = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        shell=shell,
        creationflags=_new_console_flags(),
    )
    _processes.append(proc)
    return proc


def wait_for_url(url: str, timeout: float = 60.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as resp:
                print(f"  OK {url} (HTTP {resp.status})")
                return True
        except urllib.error.HTTPError as exc:
            # API may return 401/403 when up but unauthenticated — still alive
            print(f"  OK {url} (HTTP {exc.code})")
            return True
        except Exception:
            time.sleep(1)
    print(f"  Timed out waiting for {url}")
    return False


def stop_servers() -> None:
    for proc in _processes:
        if proc.poll() is None:
            proc.terminate()
    _processes.clear()
    print("Stopped all servers started from this notebook.")
    print("(If servers opened in separate windows, close those windows manually.)")


# ── Prerequisites ──────────────────────────────────────────────────────────
missing = []
if shutil.which("npm") is None:
    missing.append("npm (install Node.js)")
if not Path(PYTHON).exists() and shutil.which("python") is None:
    missing.append("python")
if missing:
    raise RuntimeError("Missing: " + ", ".join(missing))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {PYTHON}")
print(f"npm          : {shutil.which('npm')}")

## Optional — apply database migrations

Run this cell on first setup or after pulling schema changes.

In [ ]:
RUN_MIGRATE = True  # set False to skip

if RUN_MIGRATE:
    result = subprocess.run(
        [PYTHON, "manage.py", "migrate", "--noinput"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr or "migrate complete")
    if result.returncode != 0:
        raise RuntimeError(f"migrate failed (exit {result.returncode})")
else:
    print("Skipped migrations.")

## Launch servers

Starts Django and Vite. On Windows, each runs in a **new console window**.

In [ ]:
INSTALL_NPM = not (PROJECT_ROOT / "node_modules").exists()

if INSTALL_NPM:
    print("node_modules not found — running npm install...")
    subprocess.run("npm install", cwd=PROJECT_ROOT, shell=True, check=True)

# Stop any servers started earlier in this session
stop_servers()

backend = start_process(
    f"Django API (:{BACKEND_PORT})",
    [PYTHON, "manage.py", "runserver", str(BACKEND_PORT)],
)

time.sleep(2)

frontend = start_process(
    f"Vite frontend (:{FRONTEND_PORT})",
    "npm run dev",
    shell=True,
)

print("\nWaiting for servers to become ready...")
backend_ok = wait_for_url(BACKEND_URL, timeout=90)
frontend_ok = wait_for_url(FRONTEND_URL, timeout=90)

if backend_ok and frontend_ok:
    print("\nAll servers are up.")
    webbrowser.open(FRONTEND_URL)
    print(f"Opened {FRONTEND_URL}")
else:
    print("\nOne or more servers did not respond in time.")
    print("Check the console windows for error messages.")

## Stop servers

Run when finished. If servers opened in separate windows, close those windows too.

In [ ]:
stop_servers()